# Asteroid Orbital Visual — Automated Pipeline
Run **Cell → Run All** to execute the full pipeline end-to-end:
1. Flatten raw NASA CNEOS data → `asteroids_flat.csv`
2. Verify Node.js / npm
3. Install `powerbi-visuals-tools` globally (skipped if already installed)
4. Install visual npm dependencies
5. Install the pbiviz developer certificate
6. Start the pbiviz dev server (background)
7. Launch Power BI Desktop with the project `.pbix` file
8. Print field-mapping checklist
9. Preview and profile the flattened data

In [ ]:
import subprocess, sys, os, time, threading, glob
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
VISUAL_DIR   = NOTEBOOK_DIR / 'asteroid-orbital-visual'
DATA_SCRIPT  = NOTEBOOK_DIR / 'flatten_asteroids.py'
FLAT_CSV     = NOTEBOOK_DIR / 'asteroids_flat.csv'

def run(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    return r.stdout.strip() or r.stderr.strip(), r.returncode

def ok(msg):  print(f'  ✅ {msg}')
def err(msg): print(f'  ❌ {msg}')
def info(msg):print(f'  ℹ  {msg}')

print('Paths')
print(f'  Notebook : {NOTEBOOK_DIR}')
print(f'  Visual   : {VISUAL_DIR}')
print(f'  Python   : {sys.executable}')

## Step 1 — Flatten the NASA data

In [ ]:
print('Step 1 — Flatten NASA CNEOS data')
result = subprocess.run([sys.executable, str(DATA_SCRIPT)], capture_output=True, text=True)
if result.returncode != 0:
    err('Data prep failed'); print(result.stderr); raise RuntimeError
print(result.stdout.strip())
rows = sum(1 for _ in open(FLAT_CSV)) - 1
size = FLAT_CSV.stat().st_size / 1024
ok(f'asteroids_flat.csv — {rows:,} rows, {size:.0f} KB')

## Step 2 — Verify Node.js & npm

In [ ]:
print('Step 2 — Node.js / npm')
node_ver, node_rc = run('node --version')
npm_ver,  npm_rc  = run('npm --version')
if node_rc != 0 or npm_rc != 0:
    err('Node.js not found. Install from https://nodejs.org then restart the kernel.')
    raise EnvironmentError
ok(f'Node {node_ver}  |  npm {npm_ver}')

## Step 3 — Install powerbi-visuals-tools globally

In [ ]:
print('Step 3 — pbiviz CLI')
ver, rc = run('pbiviz --version')
if rc == 0:
    ok(f'pbiviz {ver} already installed')
else:
    info('Installing powerbi-visuals-tools globally...')
    out, rc = run('npm install -g powerbi-visuals-tools')
    if rc != 0: err(out); raise RuntimeError
    ver, _ = run('pbiviz --version')
    ok(f'pbiviz {ver} installed')

## Step 4 — Install visual npm dependencies

In [ ]:
print('Step 4 — npm install')
node_modules = VISUAL_DIR / 'node_modules'
if node_modules.exists():
    ok('node_modules already present — skipping')
else:
    info('Running npm install...')
    out, rc = run('npm install', cwd=str(VISUAL_DIR))
    if rc != 0: err(out); raise RuntimeError
    ok('Dependencies installed')

## Step 5 — Developer certificate

In [ ]:
print('Step 5 — Developer certificate')
cert_path = Path.home() / 'pbiviz-certs' / 'PowerBICustomVisualTest_public.pfx'
if cert_path.exists():
    ok(f'Certificate already exists at {cert_path}')
else:
    info('Generating certificate...')
    out, rc = run('pbiviz install-cert', cwd=str(VISUAL_DIR))
    print(out)
    if cert_path.exists():
        ok('Certificate generated')
        print()
        print('  ⚠  ACTION REQUIRED — trust the certificate:')
        print('  Run this in an Admin PowerShell:')
        passphrase = [l for l in out.splitlines() if 'Passphrase' in l]
        pw = passphrase[0].split()[-1] if passphrase else 'SEE ABOVE'
        print(f"  Import-PfxCertificate -FilePath '{cert_path}' -CertStoreLocation Cert:\\CurrentUser\\Root -Password (ConvertTo-SecureString '{pw}' -Force -AsPlainText)")
    else:
        err('Certificate generation failed — check output above')

## Step 6 — Start the pbiviz dev server

In [ ]:
print('Step 6 — pbiviz dev server')

server_proc = subprocess.Popen(
    'pbiviz start',
    shell=True, cwd=str(VISUAL_DIR),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

server_lines = []
def _stream():
    for line in server_proc.stdout:
        server_lines.append(line.rstrip())

threading.Thread(target=_stream, daemon=True).start()
time.sleep(8)

errors    = [l for l in server_lines if 'error' in l.lower() and 'deprecation' not in l.lower()]
listening = any('8080' in l or 'listening' in l.lower() for l in server_lines)
compiled  = any('compiled successfully' in l for l in server_lines)

if errors:
    print('  Server output:')
    for l in server_lines[-20:]: print(f'    {l}')
    err('Errors detected — check output above')
elif listening and compiled:
    ok('Dev server running on https://localhost:8080  |  webpack compiled successfully')
else:
    info('Server starting — check output below if it takes longer than expected')
    for l in server_lines[-10:]: print(f'    {l}')

def kill_server():
    server_proc.terminate()
    print('Dev server stopped.')

## Step 7 — Launch Power BI Desktop

In [ ]:
print('Step 7 — Launch Power BI Desktop')

# Candidate install paths (covers Store, installer, and per-user installs)
pbi_candidates = [
    r'C:\Program Files\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    r'C:\Program Files (x86)\Microsoft Power BI Desktop\bin\PBIDesktop.exe',
    str(Path.home() / 'AppData\Local\Microsoft\WindowsApps\PBIDesktop.exe'),
]
# Also search WindowsApps for Store version
pbi_candidates += glob.glob(
    str(Path.home() / 'AppData/Local/Microsoft/WindowsApps/**/PBIDesktop.exe'),
    recursive=True
)

pbi_exe = next((p for p in pbi_candidates if Path(p).exists()), None)

# Look for an existing .pbix to open
pbix_files = list(NOTEBOOK_DIR.glob('*.pbix'))
pbix_arg   = str(pbix_files[0]) if pbix_files else None

if pbi_exe:
    cmd = [pbi_exe] + ([pbix_arg] if pbix_arg else [])
    subprocess.Popen(cmd)
    if pbix_arg:
        ok(f'Launched Power BI Desktop with {Path(pbix_arg).name}')
    else:
        ok('Launched Power BI Desktop (no .pbix found — will open blank)')
        info('Load asteroids_flat.csv via Home → Get data → Text/CSV')
else:
    err('Power BI Desktop not found in standard locations.')
    print('  Install from https://powerbi.microsoft.com/desktop or the Microsoft Store.')
    print('  Then re-run this cell.')

## Step 8 — Field mapping checklist
Drag these columns from `asteroids_flat.csv` into the **Data Fields** bucket of the Developer Visual:

In [ ]:
import pandas as pd

FIELD_MAP = [
    ('name',                  'Asteroid identity'),
    ('short_name',            'Label in tooltips'),
    ('potentially_hazardous', 'Red color + glow pulse'),
    ('diameter_max_m',        'Dot size'),
    ('semi_major_axis',       'Orbit radius (AU)'),
    ('eccentricity',          'Orbit shape'),
    ('perihelion_argument',   'Orbit orientation'),
    ('miss_distance_au',      'Closest approach distance (tooltip)'),
    ('close_approach_date',   'Approach date (tooltip)'),
    ('velocity_km_s',         'Approach speed (tooltip)'),
    ('orbiting_body',         'Which planet approached'),
    ('magnitude',             'Brightness (tooltip)'),
    ('orbit_class_type',      'Class label (tooltip)'),
]

df_map = pd.DataFrame(FIELD_MAP, columns=['Column in asteroids_flat.csv', 'What it drives'])
df_map.index += 1

print('Power BI field mapping')
print('=' * 60)
for col, role in FIELD_MAP:
    print(f'  {col:<30} →  {role}')
print()
print('Formatting pane options (in the visual panel):')
print('  Show All 8 Planets    — toggle outer planets on/off')
print('  Animation Speed       — days simulated per frame (default 5)')
print('  Hazardous Only        — filter to red asteroids only')

## Step 9 — Data profile

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(FLAT_CSV, low_memory=False)
df['miss_distance_au'] = pd.to_numeric(df['miss_distance_au'], errors='coerce')
df['velocity_km_s']    = pd.to_numeric(df['velocity_km_s'],    errors='coerce')
df['diameter_max_m']   = pd.to_numeric(df['diameter_max_m'],   errors='coerce')
df['close_approach_date'] = pd.to_datetime(df['close_approach_date'], errors='coerce')

print(f'Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Asteroids     : {df["name"].nunique():,} unique')
print(f'Hazardous     : {(df["potentially_hazardous"]=="True").sum():,} approaches by hazardous objects')
print(f'Date range    : {df["close_approach_date"].min().date()} → {df["close_approach_date"].max().date()}')
print(f'Orbiting body : {df["orbiting_body"].value_counts().to_dict()}')
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('NASA CNEOS Asteroid Close Approaches', fontsize=13, fontweight='bold')

# Miss distance distribution
close = df[df['miss_distance_au'] < 0.5]
axes[0].hist(close['miss_distance_au'].dropna(), bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Miss Distance Distribution (< 0.5 AU)')
axes[0].set_xlabel('Miss Distance (AU)')
axes[0].set_ylabel('Approach count')

# Approaches per decade
df['decade'] = (df['close_approach_date'].dt.year // 10 * 10).dropna()
decade_counts = df['decade'].value_counts().sort_index()
axes[1].bar(decade_counts.index.astype(str), decade_counts.values, color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].set_title('Close Approaches by Decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Hazardous vs non-hazardous velocity
haz     = df[df['potentially_hazardous'] == 'True']['velocity_km_s'].dropna()
non_haz = df[df['potentially_hazardous'] == 'False']['velocity_km_s'].dropna()
axes[2].hist(non_haz, bins=40, alpha=0.6, color='gray',   label='Non-hazardous', edgecolor='white', linewidth=0.2)
axes[2].hist(haz,     bins=40, alpha=0.8, color='crimson', label='Hazardous',     edgecolor='white', linewidth=0.2)
axes[2].set_title('Relative Velocity by Hazard Status')
axes[2].set_xlabel('Velocity (km/s)')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / 'assets' / 'data_profile.png', dpi=120, bbox_inches='tight')
plt.show()
print('Profile chart saved to assets/data_profile.png')

In [ ]:
# Top 10 closest Earth approaches in the dataset
earth = df[df['orbiting_body'] == 'Earth'].copy()
earth['miss_distance_au'] = pd.to_numeric(earth['miss_distance_au'], errors='coerce')
closest = (earth.sort_values('miss_distance_au')
               [['name', 'close_approach_date', 'miss_distance_au', 'velocity_km_s', 'potentially_hazardous']]
               .drop_duplicates('name')
               .head(10)
               .reset_index(drop=True))
closest.index += 1
print('Top 10 closest Earth approaches (one per asteroid):')
closest

In [ ]:
# Run this cell to stop the dev server when done
# kill_server()